In [0]:
import pandas as pd 

In [0]:
#Leyendo la tabla de características generales del catálogo
cgse=spark.read.format("delta").table("workspace.electiva.cgse")
ft = spark.read.format("delta").table("workspace.electiva.ft")
     

In [0]:
PET = ft.filter(ft['PET'] == 1).agg({'FEX_C18': 'sum'}).collect()[0][0]
print(f'La población que está en edad de trabajar es de {PET:,.0f} personas')
     
cgse_pd = cgse.toPandas()
PT = cgse_pd['FEX_C18'].sum()
PNET = PT - PET
print(f'La población que no está en edad de trabajar es de {PNET:,.0f} personas')
 
PEA = ft.filter(ft['FT'] == 1).agg({'FEX_C18': 'sum'}).collect()[0][0]
print(f'La población económicamente activa es de {PEA:,.0f} personas')
     
PEI = PET - PEA
print(f'La población económicamente inactiva es de {PEI:,.0f} personas')
     
ocupados = spark.read.format("delta").table("workspace.electiva.ocupados").toPandas()
O = ocupados[ocupados['OCI']==1]['FEX_C18'].sum()
TO = O/PEA*100
TO = O/PET*100
print(f'El porcentaje de ocupados es de {TO:.2f}%')
     
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados").toPandas()
D = desempleados[desempleados['DSI']==1]['FEX_C18'].sum()  # Cell 23
TD = D/PEA*100
print(f'El porcentaje de desempleados es de {TD:.2f}%')
     

In [0]:
#2 Calcule las tasas de desempleo para hombres y mujeres en todos los departamentos de Colombia a Junio de 2026, ¿cuáles son los departamentos con mayores y menores tasas de desempleo?
import pandas as pd

# Leyendo la tabla de características generales
ft = spark.read.format("delta").table("workspace.electiva.ft")
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados")
cgse = spark.read.format("delta").table("workspace.electiva.cgse")

# Convertir el DataFrame de Spark a Pandas
ft = ft.toPandas()
desempleados = desempleados.toPandas()
cgse = cgse.toPandas()

# Primero hay que anexar la P3271 a la tabla de desempleados
desempleados = pd.merge(desempleados,cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271', 'P6040']],on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'],how='left')

ft = pd.merge(ft,cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271', 'P6040']],on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'],how='left')
     
     

In [0]:
# desempleados_dpto -> cantidad de desempleados por departamento
desempleados_dpto = desempleados[desempleados['DSI'] == 1].groupby(['P3271', 'DPTO'])['FEX_C18'].sum()
desempleados_dpto
     

In [0]:
# Cantidad de hombres y mujeres económicamente activos por departamento
pea_dpto_genero = ft[ft['FT'] == 1].groupby(['P3271', 'DPTO'])['FEX_C18'].sum()

In [0]:
tabla_desempleo = pd.DataFrame({'PEA': pea_dpto_genero,'DESEMPLEADOS': desempleados_dpto})

In [0]:
tabla_desempleo

tabla_desempleo['TD'] = (tabla_desempleo['DESEMPLEADOS'] / tabla_desempleo['PEA']) * 100
tabla_desempleo.head(100)

In [0]:
# 4. Diccionarios de mapeo
dpto_nombre = {5: 'ANTIOQUIA', 8: 'ATLANTICO', 11: 'BOGOTA', 13: 'BOLIVAR', 15: 'BOYACA',17: 'CALDAS', 18: 'CAQUETA', 19: 'CAUCA', 20: 'CESAR', 23: 'CORDOBA',25: 'CUNDINAMARCA', 27: 'CHOCO', 41: 'HUILA', 44: 'GUAJIRA', 47: 'MAGDALENA',50: 'META', 52: 'NARIÑO', 54: 'NORTE DE SANTANDER', 63: 'QUINDIO',66: 'RISARALDA', 68: 'SANTANDER', 70: 'SUCRE', 73: 'TOLIMA',76: 'VALLE DEL CAUCA', 81: 'ARAUCA', 85: 'CASANARE', 86: 'PUTUMAYO',88: 'SAN ANDRES', 91: 'AMAZONAS', 94: 'GUAINIA', 95: 'GUAVIARE',97: 'VAUPES', 99: 'VICHADA'}

sexo = {1: 'Hombre', 2: 'Mujer'}

In [0]:
tabla_desempleo = tabla_desempleo.reset_index()

tabla_desempleo['Departamento'] = tabla_desempleo['DPTO'].map(dpto_nombre)
tabla_desempleo['Sexo'] = tabla_desempleo['P3271'].map(sexo)

In [0]:
tabla_desempleo = tabla_desempleo[['Departamento', 'Sexo', 'PEA', 'DESEMPLEADOS', 'TD']]
     

In [0]:
# Ordenar por dos variables de mayor a menor (descendente)
tabla_desempleo = tabla_desempleo.sort_values(by=['Sexo', 'TD'],ascending=False)
tabla_desempleo


In [0]:
tabla_desempleo[tabla_desempleo['Sexo'] == 'Mujer']

In [0]:
tabla_desempleo[tabla_desempleo['Sexo'] == 'Hombre']

In [0]:
# Función para mapear la edad (P6040) a grupos de edad
def mapear_grupo_edad(edad):
    if 18 <= edad < 24:
        return '18 a 24'
    elif 24 <= edad < 30:
        return '24 a 30'
    elif 30 <= edad < 40:
        return '30 a 40'
    elif 40 <= edad < 50:
        return '40 a 50'
    elif edad >= 60:
        return '60 en adelante'
    else:
        return None

# Aplicar el mapeo a ambos DataFrames
desempleados['GRUPO_EDAD'] = desempleados['P6040'].map(mapear_grupo_edad)
ft['GRUPO_EDAD'] = ft['P6040'].map(mapear_grupo_edad)

In [0]:
# Desempleados por grupo de edad y departamento (reutilizando código de pregunta 2)
desempleados_dpto_edad = desempleados[desempleados['DSI'] == 1].groupby(['GRUPO_EDAD', 'DPTO'])['FEX_C18'].sum()

# PEA por grupo de edad y departamento
pea_dpto_edad = ft[ft['FT'] == 1].groupby(['GRUPO_EDAD', 'DPTO'])['FEX_C18'].sum()

# Crear tabla de desempleo (reutilizando estructura de pregunta 2)
tabla_desempleo_edad = pd.DataFrame({'PEA': pea_dpto_edad, 'DESEMPLEADOS': desempleados_dpto_edad})

In [0]:
# Calcular tasa de desempleo (reutilizando código de pregunta 2)
tabla_desempleo_edad['TD'] = (tabla_desempleo_edad['DESEMPLEADOS'] / tabla_desempleo_edad['PEA']) * 100

# Mapear nombres de departamentos (reutilizando dpto_nombre de pregunta 2)
tabla_desempleo_edad = tabla_desempleo_edad.reset_index()
tabla_desempleo_edad['Departamento'] = tabla_desempleo_edad['DPTO'].map(dpto_nombre)

# Seleccionar y ordenar columnas
tabla_desempleo_edad = tabla_desempleo_edad[['Departamento', 'GRUPO_EDAD', 'PEA', 'DESEMPLEADOS', 'TD']]
tabla_desempleo_edad = tabla_desempleo_edad.sort_values(by=['Departamento', 'TD'], ascending=[True, False])

display(tabla_desempleo_edad)

In [0]:
# Grupo de edad con mayor tasa de desempleo por departamento
max_td = tabla_desempleo_edad.loc[tabla_desempleo_edad.groupby('Departamento')['TD'].idxmax()]
max_td = max_td[['Departamento', 'GRUPO_EDAD', 'TD']].reset_index(drop=True)
max_td.columns = ['Departamento', 'Mayor grupo de edad', 'Mayor TD']

# Grupo de edad con menor tasa de desempleo por departamento
min_td = tabla_desempleo_edad.loc[tabla_desempleo_edad.groupby('Departamento')['TD'].idxmin()]
min_td = min_td[['Departamento', 'GRUPO_EDAD', 'TD']].reset_index(drop=True)
min_td.columns = ['Departamento', 'Menor grupo de edad', 'Menor TD']

# Combinar resultados
resultado_edad = pd.merge(max_td, min_td, on='Departamento')
display(resultado_edad)

In [0]:
#4. ¿Cuáles son los 3 departamentos con menor tasa de desempleo de mujeres entre los 18 y los 25 años ?
# Mujeres desempleadas entre 18 y 25 años por departamento
desempleadas_jovenes = desempleados[
    (desempleados['DSI'] == 1) &
    (desempleados['P3271'] == 2) &
    (desempleados['P6040'].between(18, 25))
].groupby('DPTO')['FEX_C18'].sum()

# Mujeres económicamente activas entre 18 y 25 años por departamento
pea_mujeres_jovenes = ft[
    (ft['FT'] == 1) &
    (ft['P3271'] == 2) &
    (ft['P6040'].between(18, 25))
].groupby('DPTO')['FEX_C18'].sum()

# Crear tabla con los resultados
tabla_mujeres_jovenes = pd.DataFrame({
    'PEA': pea_mujeres_jovenes,
    'DESEMPLEADAS': desempleadas_jovenes
})

# Completar con cero donde no se registraron desempleadas
tabla_mujeres_jovenes['DESEMPLEADAS'] = (
    tabla_mujeres_jovenes['DESEMPLEADAS'].fillna(0)
)

# Conservar departamentos con población económicamente activa
tabla_mujeres_jovenes = tabla_mujeres_jovenes[
    tabla_mujeres_jovenes['PEA'] > 0
].copy()

# Calcular la tasa de desempleo
tabla_mujeres_jovenes['TD'] = (
    tabla_mujeres_jovenes['DESEMPLEADAS'] /
    tabla_mujeres_jovenes['PEA']
) * 100

# Agregar los nombres de los departamentos
tabla_mujeres_jovenes = tabla_mujeres_jovenes.reset_index()
tabla_mujeres_jovenes['Departamento'] = (
    tabla_mujeres_jovenes['DPTO'].map(dpto_nombre)
)

# Excluir Bogotá, porque es un distrito y se solicitan departamentos
tabla_mujeres_jovenes = tabla_mujeres_jovenes[
    tabla_mujeres_jovenes['DPTO'] != 11
]

# Seleccionar los tres departamentos con menor tasa de desempleo
resultado_punto4 = tabla_mujeres_jovenes.nsmallest(
    3, 'TD'
)[['Departamento', 'PEA', 'DESEMPLEADAS', 'TD']].reset_index(drop=True)

display(resultado_punto4)

# Respuesta escrita con los resultados obtenidos
print('Los tres departamentos con menor tasa de desempleo '
      'de mujeres entre 18 y 25 años son:')

for _, fila in resultado_punto4.iterrows():
    print(f"- {fila['Departamento']}: {fila['TD']:.2f}%.")

print(
    '\nEstas tasas representan el porcentaje de mujeres desocupadas '
    'dentro de las mujeres económicamente activas de 18 a 25 años '
    'en cada departamento. Una tasa menor no implica necesariamente '
    'mayor participación laboral ni mejores condiciones de empleo.')

In [0]:
#5. Calcule el promedio de ingresos laborales (variable INGLABO) de las personas residentes en los departamentos de Risaralda, Caldas , Quindío, Antioquia y Valle del Cauca.
# Leer la tabla de ocupados
ocupados = spark.read.format("delta").table(
    "workspace.electiva.ocupados"
).toPandas()

# Departamentos: Risaralda, Caldas, Quindío, Antioquia y Valle del Cauca
departamentos = [66, 17, 63, 5, 76]

# Seleccionar personas ocupadas de los departamentos solicitados
ingresos_dpto = ocupados[
    (ocupados['OCI'] == 1) &
    (ocupados['DPTO'].isin(departamentos))
].copy()

# Convertir ingresos y factor de expansión a valores numéricos
ingresos_dpto['INGLABO'] = pd.to_numeric(
    ingresos_dpto['INGLABO'], errors='coerce'
)

ingresos_dpto['FEX_C18'] = pd.to_numeric(
    ingresos_dpto['FEX_C18'], errors='coerce'
)

# Conservar ingresos registrados y factores positivos
ingresos_dpto = ingresos_dpto[
    ingresos_dpto['INGLABO'].notna() &
    (ingresos_dpto['FEX_C18'] > 0)
].copy()

# Calcular el ingreso ponderado
ingresos_dpto['INGRESO_PONDERADO'] = (
    ingresos_dpto['INGLABO'] * ingresos_dpto['FEX_C18']
)

# Sumar los ingresos ponderados y los factores por departamento
tabla_ingresos = ingresos_dpto.groupby('DPTO')[
    ['INGRESO_PONDERADO', 'FEX_C18']
].sum().reset_index()

# Calcular el promedio de ingresos laborales
tabla_ingresos['Promedio INGLABO'] = (
    tabla_ingresos['INGRESO_PONDERADO'] /
    tabla_ingresos['FEX_C18']
)

# Agregar nombres de departamentos
tabla_ingresos['Departamento'] = (
    tabla_ingresos['DPTO'].map(dpto_nombre)
)

# Seleccionar columnas y ordenar de mayor a menor ingreso
resultado_punto5 = tabla_ingresos[
    ['Departamento', 'Promedio INGLABO']
].sort_values(
    by='Promedio INGLABO', ascending=False
).reset_index(drop=True)

display(resultado_punto5)